# Realistic Face Generator 

In this notebook, we develop a Realistic Face Generator using a Generative Adversarial Network (GAN) trained on the CelebA dataset. The model learns to generate a realistic human faces by capturing complex patterns and features from the dataset through adversarial training.

For more details on Celeba dataset visit: https://www.kaggle.com/datasets/jessicali9530/celeba-dataset/data

# Pre-requisites

To support features of this notebook with CoreAI, we need to install some libraries that are not pre-installed but are required for this notebook. 

## Create and Activate the Virtual Environment:
Open your terminal or command prompt within the Jupyter notebook. Navigate via `File -> New -> Terminal`.
Type `bash` to access a shell compatible with the following commands.
Navigate to the project directory where you want to set up the environment (where this notebook is located):

```bash
export PROJECT_NAME="Realistic_Face_Generator"
export PIP_CACHE_DIR=`pwd`/.cache/pip
mkdir -p $PIP_CACHE_DIR
python -m venv --system-site-packages myvenv
source myvenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=${PROJECT_NAME}_myvenv --display-name="Python (${PROJECT_NAME}_myvenv)"
echo ""; echo "Before continuing load the created Python kernel: Python (${PROJECT_NAME}_myvenv)"
```

Load the Python kernel described above before running the cell below (it might take a few seconds for the kernel to appear in the list of kernels).

The following will set the folder location for download so that they are local to the running container, to provide cache.

In [ ]:
import os
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from torchvision.utils import save_image, make_grid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
file_path = "celeba-dataset.zip"
download_url = "https://www.kaggle.com/api/v1/datasets/download/jessicali9530/celeba-dataset"

if not os.path.exists(file_path):
    print("File not found. Downloading celeba-dataset.zip...")
    subprocess.run([
        "curl", "-L", "-o", file_path, download_url
    ])
else:
    print("File celeba-dataset.zip already exists. Skipping download.")

In [ ]:
folder_path = "img_align_celeba"
zip_file = "celeba-dataset.zip"

if not os.path.exists(folder_path):
    print(f"Folder '{folder_path}' not found. Unzipping {zip_file}...")
    subprocess.run(["unzip", "-q", zip_file])
else:
    print(f"Folder '{folder_path}' already exists. Skipping unzip.")

## Hyperparameter Settings

**Quick Run:**  
For a quick test run, set:  
- `image_size = 64`  
- `epochs = 10`

**Note:**  
To achieve good image generation quality, set a reasonable number of epochs based on the image size:  
- **Larger image sizes** generally require **more training epochs**.  
- **Smaller image sizes** can converge with **fewer epochs**.

Adjust accordingly depending on your hardware and desired output quality.

In [ ]:
image_size = int(input("Enter the image size (32, 64, 128, or 256): "))
epochs = int(input("Enter the number of epochs: "))
print(f"Selected image size: {image_size}x{image_size}")

In [ ]:
z_dim = 100
channels = 3
beta1 = 0.5

if image_size not in [32, 64, 128, 256]:
    raise ValueError(f"Image size {image_size} not supported. Use 32, 64, 128, or 256.")
        

if image_size <= 64:
    batch_size = 128
    learning_rate = 0.0002
    num_epochs = max(10,epochs)
    label_smoothing = 0.0
    gradient_clip = None
    d_lr_multiplier = 1.0
elif image_size == 128:
    batch_size = 64        
    learning_rate = 0.0001 
    num_epochs = max(15,epochs)      
    label_smoothing = 0.1    
    gradient_clip = 1.0        
    d_lr_multiplier = 0.5    
else: 
    batch_size = 32       
    learning_rate = 0.00005 
    num_epochs = max(20,epochs)        
    label_smoothing = 0.15     
    gradient_clip = 0.5        
    d_lr_multiplier = 0.3      

print(f"Optimized hyperparameters for {image_size}x{image_size}:")
print(f"  batch_size: {batch_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  num_epochs: {num_epochs}")
print(f"  label_smoothing: {label_smoothing}")
print(f"  gradient_clip: {gradient_clip}")
print(f"  d_lr_multiplier: {d_lr_multiplier}")

### Generator Architecture

The `Generator` class defines the generator network of our GAN. Its purpose is to map a random noise vector from a latent space (typically a vector of size `z_dim = 100`) to a realistic-looking face image.

We use a **Deep Convolutional Generative Adversarial Network (DCGAN)**-style architecture, where transposed convolutional layers are used to upsample the input noise into a high-resolution image.

**Adaptive Architecture**: The generator automatically adapts its depth based on the target `image_size`:
- **32×32**: 3 layers (4→8→16→32)
- **64×64**: 4 layers (4→8→16→32→64) 
- **128×128**: 5 layers (4→8→16→32→64→128)
- **256×256**: 6 layers (4→8→16→32→64→128→256)

#### Architecture Details:
- **Input**: A latent vector of shape `(z_dim, 1, 1)` (typically sampled from a standard normal distribution).
- **Layers**:
  - `ConvTranspose2d`: Upsamples the input and increases spatial resolution.
  - `BatchNorm2d`: Normalizes activations and stabilizes training.
  - `ReLU`: Applies non-linearity after each upsampling layer.
  - Final `ConvTranspose2d` layer outputs a 3-channel image (RGB) followed by `Tanh` to scale pixel values to `[-1, 1]`.

#### Output:
- A generated RGB image of shape `(3, image_size, image_size)` — a realistic-looking human face.

This generator learns to synthesize convincing face images through adversarial training by trying to fool the discriminator into believing the generated image is real.


In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, channels=3, features=64, image_size=64):
        super(Generator, self).__init__()
        
        self.image_size = image_size
        
        layers_needed = int(np.log2(image_size)) - 2  
        
        layers = []
        
        current_features = features * (2 ** layers_needed)
        layers.extend([
            nn.ConvTranspose2d(z_dim, current_features, 4, 1, 0, bias=False),
            nn.BatchNorm2d(current_features),
            nn.ReLU(True)
        ])
        
        for i in range(layers_needed - 1):
            next_features = current_features // 2
            layers.extend([
                nn.ConvTranspose2d(current_features, next_features, 4, 2, 1, bias=False),
                nn.BatchNorm2d(next_features),
                nn.ReLU(True)
            ])
            current_features = next_features
        
        layers.extend([
            nn.ConvTranspose2d(current_features, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        ])
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

### Discriminator Architecture

The `Discriminator` class defines the discriminator network of our GAN. Its role is to distinguish between **real images** from the CelebA dataset and **fake images** generated by the generator.

We use a **Deep Convolutional GAN (DCGAN)**-inspired architecture, composed of convolutional layers that progressively **downsample** the input image and learn hierarchical features.

**Adaptive Architecture**: The discriminator automatically adapts its depth based on the input `image_size`:
- **32×32**: 3 layers (32→16→8→4→1)
- **64×64**: 4 layers (64→32→16→8→4→1)
- **128×128**: 5 layers (128→64→32→16→8→4→1)  
- **256×256**: 6 layers (256→128→64→32→16→8→4→1)

#### Architecture Details:
- **Input**: An RGB image of shape `(3, image_size, image_size)` (either real or generated).
- **Layers**:
  - `Conv2d`: Reduces spatial dimensions while increasing feature channels.
  - `BatchNorm2d`: Normalizes outputs to stabilize and accelerate training (used after all but the first convolution).
  - `LeakyReLU`: Introduces non-linearity with a small slope for negative values to avoid dying neurons.
  - The **final convolutional layer** reduces the output to a single value.
  - `Sigmoid`: Outputs a probability between `0` and `1`, indicating how "real" the image is.

#### Output:
- A scalar value between `0` and `1` for each input image, where:
  - `1` → real image
  - `0` → generated (fake) image

This network learns to **penalize** the generator when it produces unrealistic images and improves over time as part of the adversarial training loop.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, channels=3, features=64, image_size=64):
        super(Discriminator, self).__init__()
        
        self.image_size = image_size
        
        layers_needed = int(np.log2(image_size)) - 2  # 
        
        layers = []
        
        layers.extend([
            nn.Conv2d(channels, features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True)
        ])
        
        current_features = features
        
        for i in range(layers_needed - 1):
            next_features = current_features * 2
            layers.extend([
                nn.Conv2d(current_features, next_features, 4, 2, 1, bias=False),
                nn.BatchNorm2d(next_features),
                nn.LeakyReLU(0.2, inplace=True)
            ])
            current_features = next_features
        
        layers.extend([
            nn.Conv2d(current_features, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        ])
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).view(-1, 1).squeeze(1)

### Weight Initialization

The `weights_init` function initializes the weights of the Generator and Discriminator networks using a custom strategy recommended in the **DCGAN** paper.

Proper initialization is crucial for stable and efficient GAN training.

#### How it works:
- It checks the **type of layer** (using `classname`) and initializes weights accordingly:
  - For **Convolutional layers** (`Conv`), weights are initialized from a normal distribution with:
    - Mean = `0.0`
    - Standard deviation = `0.02`
  - For **Batch Normalization layers** (`BatchNorm`):
    - Weights are initialized from a normal distribution with:
      - Mean = `1.0`
      - Standard deviation = `0.02`
    - Biases are initialized to `0`

This approach helps in avoiding **vanishing or exploding gradients**, especially during the early phases of training.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

### Data Loader for CelebA Dataset

The `get_celeba_loader` function creates and returns a **PyTorch DataLoader** for the CelebA dataset, which is essential for feeding images into the GAN during training.

#### Function Overview:
This function:
1. Applies necessary **image transformations**,
2. Loads the dataset from a specified directory, and
3. Returns a `DataLoader` that serves batches of preprocessed images during training.

#### Transformations Applied:
- `Resize((image_size, image_size))`: Resizes all images to the target resolution (default: `64×64`).
- `ToTensor()`: Converts images to PyTorch tensors with shape `(C, H, W)` and values in `[0, 1]`.
- `Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))`: Normalizes image pixels to the range `[-1, 1]`, which is important for GANs using `Tanh` in the output layer of the generator.

#### Parameters:
- `data_root`: Path to the directory containing images (should match the `img_align_celeba` folder).
- `batch_size`: Number of images per batch (default: `128`).
- `image_size`: Target spatial dimensions of each image (default: `64`).

#### Returns:
- A `DataLoader` object that yields batches of normalized images from the CelebA dataset, shuffled at each epoch.

This function ensures consistent preprocessing and efficient loading for GAN training.


In [ ]:
def get_celeba_loader(data_root, batch_size=128, image_size=64):
    
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
    ])
    
    dataset = torchvision.datasets.ImageFolder(root=data_root, transform=transform)
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    return dataloader

### GAN Training Loop

The `train_gan` function encapsulates the entire training process of the Generative Adversarial Network (GAN) using the CelebA dataset.

This function trains both the **Generator** and **Discriminator** networks in an adversarial setup, where:
- The **Generator** learns to produce realistic images from random noise.
- The **Discriminator** learns to distinguish between real images and generated (fake) ones.

#### Key Components:
- **Model Instantiation**:
  - `Generator` and `Discriminator` are instantiated and moved to the specified device (CPU/GPU).
  - Weights are initialized using the custom `weights_init` function.

- **Loss Function**:
  - `BCELoss` (Binary Cross-Entropy Loss) is used for both networks.
    - Real images are labeled as `1`, fake images as `0`.

- **Optimizers**:
  - Adam optimizers are used for both networks with common hyperparameters from DCGAN literature (`betas=(0.5, 0.999)`).

- **Fixed Noise**:
  - A fixed noise tensor is used to generate a consistent grid of images after every 5 epochs to monitor training progress.

#### Training Steps:
1. **Discriminator Training**:
   - Computes loss on real images (`loss_D_real`).
   - Generates fake images from random noise and computes loss on them (`loss_D_fake`).
   - Combines both and updates the discriminator (`loss_D`).

2. **Generator Training**:
   - Generates fake images and feeds them to the discriminator.
   - Computes how well these fake images fooled the discriminator (`loss_G`).
   - Updates the generator to improve its realism.

3. **Logging and Saving**:
   - Loss values are stored for visualization later.
   - Progress is printed every 100 batches.
   - Every 5 epochs, generated images are saved using `save_image`.

#### Returns:
- The trained **Generator** and **Discriminator** models.
- Lists of **Generator losses (`G_losses`)** and **Discriminator losses (`D_losses`)** for visualization and analysis.

This training loop is the heart of the GAN framework, where both networks continuously evolve to outperform each other, resulting in increasingly realistic image generation.


In [ ]:
def train_gan(dataloader, num_epochs=50):
    generator = Generator(z_dim, channels, image_size=image_size).to(device)
    discriminator = Discriminator(channels, image_size=image_size).to(device)
    
    generator.apply(weights_init)
    discriminator.apply(weights_init)
    
    criterion = nn.BCELoss()
    
    d_lr = learning_rate * d_lr_multiplier
    g_lr = learning_rate
    
    optimizer_G = optim.Adam(generator.parameters(), lr=g_lr, betas=(beta1, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=d_lr, betas=(beta1, 0.999))

    fixed_noise = torch.randn(64, z_dim, 1, 1, device=device)
    
    generator.train()
    discriminator.train()
    
    G_losses = []
    D_losses = []
    
    print(f"Starting Training with stabilized parameters for {image_size}x{image_size}:")
    print(f"  Generator LR: {g_lr:.6f}")
    print(f"  Discriminator LR: {d_lr:.6f}")
    print(f"  Label smoothing: {label_smoothing}")
    
    
    for epoch in range(num_epochs):
        for i, (real_images, _) in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)
            real_labels = torch.ones(batch_size, device=device) - label_smoothing * torch.rand(batch_size, device=device)
            fake_labels = torch.zeros(batch_size, device=device) + label_smoothing * torch.rand(batch_size, device=device)

            optimizer_D.zero_grad()

            output_real = discriminator(real_images)
            loss_D_real = criterion(output_real, real_labels)

            noise = torch.randn(batch_size, z_dim, 1, 1, device=device)
            fake_images = generator(noise)
            output_fake = discriminator(fake_images.detach())
            loss_D_fake = criterion(output_fake, fake_labels)
            
            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            
            if gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(discriminator.parameters(), gradient_clip)
            
            optimizer_D.step()
            optimizer_G.zero_grad()
            
            output_fake = discriminator(fake_images)
            hard_real_labels = torch.ones(batch_size, device=device)
            loss_G = criterion(output_fake, hard_real_labels)
            
            loss_G.backward()
            
            if gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(generator.parameters(), gradient_clip)
            
            optimizer_G.step()
            
            G_losses.append(loss_G.item())
            D_losses.append(loss_D.item())
            
            if i % 100 == 0:
                d_real_mean = output_real.mean().item()
                d_fake_mean = output_fake.mean().item()
                
                print(f'[{epoch}/{num_epochs}][{i}/{len(dataloader)}] '
                      f'Loss_D: {loss_D.item():.4f} Loss_G: {loss_G.item():.4f} '
                      f'D(x): {d_real_mean:.4f} D(G(z)): {d_fake_mean:.4f}')
                
        if epoch % 5 == 0:
            with torch.no_grad():
                fake_images = generator(fixed_noise)
                save_image(fake_images, f'generated_faces_epoch_{epoch}.png', normalize=True, nrow=8)
                print(f"Saved sample images for epoch {epoch}")
    
    print("\n Training completed!")
    return generator, discriminator, G_losses, D_losses

### Generate and Display Faces

The `generate_faces` function uses a trained **Generator** model to produce a set of synthetic face images from random noise.

#### How it works:
- Sets the generator to evaluation mode (`eval()`) to disable dropout and batch normalization updates.
- Generates `num_faces` latent vectors sampled from a standard normal distribution.
- Feeds these vectors through the generator to produce fake face images.
- **Denormalizes** the images by scaling pixel values from `[-1, 1]` back to `[0, 1]` for proper visualization.
- Uses `make_grid` to arrange generated images into a grid.
- Displays the grid using Matplotlib without axes and adds a title.

#### Output:
- A grid of generated face images displayed inline, allowing visual inspection of the generator’s current performance.

This function is useful for monitoring how realistic and diverse the generated faces are at any stage of training.


In [ ]:
def generate_faces(generator, num_faces=16):
    generator.eval()
    with torch.no_grad():
        noise = torch.randn(num_faces, z_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_images = (fake_images + 1) / 2
        grid = make_grid(fake_images, nrow=4, padding=2)
        plt.figure(figsize=(12, 12))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title('Generated Faces')
        plt.show()

### Plot Training Losses

The `plot_losses` function visualizes the training progress by plotting the loss values of the Generator and Discriminator over time.

#### Details:
- Plots the **Generator loss** and **Discriminator loss** on the same graph.
- The x-axis represents the training iterations (batches).
- The y-axis represents the loss value.
- A legend distinguishes between the two loss curves.
- Helps identify trends such as:
  - Whether the generator is improving (loss decreasing).
  - How well the discriminator is learning to differentiate real vs. fake.
  - Potential issues like mode collapse or unstable training.

This visualization is crucial for diagnosing and tuning the GAN training process.


In [ ]:
def plot_losses(G_losses, D_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(G_losses, label='Generator Loss')
    plt.plot(D_losses, label='Discriminator Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Losses')
    plt.show()

### Main Execution Block

This section orchestrates the full workflow of our Realistic Face Generator project:

1. **Dataset Loading**  
   - Sets the path to the CelebA dataset directory (`img_align_celeba`).
   - Loads the dataset using the `get_celeba_loader` function.
   - Checks if the dataset is loaded successfully; prompts to download if missing.

2. **Training the GAN**  
   - Starts training the Generator and Discriminator networks with the specified number of epochs.
   - Tracks and records training losses for both networks.

3. **Saving Models**  
   - After training, saves the learned weights of both Generator and Discriminator to disk for future use or inference.

4. **Generating Sample Faces**  
   - Generates and displays a grid of 16 synthetic faces from the trained Generator to visually assess performance.

5. **Plotting Training Losses**  
   - Visualizes the Generator and Discriminator loss curves over training iterations to analyze training stability and progress.

6. **Completion Message**  
   - Prints a confirmation once training and generation are completed successfully.

This block is the entry point for running the entire pipeline end-to-end, combining dataset preparation, model training, saving, and visualization.


In [ ]:
if __name__ == "__main__":
    data_root = "img_align_celeba" 

    print("Loading CelebA dataset...")
    dataloader = get_celeba_loader(data_root, batch_size, image_size)
    
    if dataloader is None:
        print("Please download the CelebA dataset and set the correct path.")
        exit()
    
    print(f"Dataset loaded. Number of batches: {len(dataloader)}")
    
    generator, discriminator, G_losses, D_losses = train_gan(dataloader, num_epochs)
    
    torch.save(generator.state_dict(), 'face_generator.pth')
    torch.save(discriminator.state_dict(), 'face_discriminator.pth')

    generate_faces(generator, 16)
    
    plot_losses(G_losses, D_losses)
    
    print("Training completed! Generated faces saved as images.")

## Inferencing

### Load Trained Generator and Generate Faces

The `load_and_generate` function allows you to load a previously trained **Generator** model from disk and generate new synthetic face images without retraining.

#### How it works:
- Instantiates a new Generator model and loads the saved weights from `'face_generator.pth'`.
- Moves the model to the appropriate device.
- Calls the `generate_faces` function to produce and display the specified number of images.

In [ ]:
def load_and_generate(num_images):
    generator = Generator(z_dim, channels, image_size=image_size).to(device)
    generator.load_state_dict(torch.load('face_generator.pth', map_location=device))
    generate_faces(generator, num_images)

In [ ]:
num_images = int(input("Enter the number of face images you want to create: "))
load_and_generate(num_images)